# connecting python with sql

In [1]:
import pyodbc

In [9]:
server=r"localhost\SQLEXPRESS"
database="covid_19"
connection_string=(
    "DRIVER={ODBC Driver 18 for SQL SERVER};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

try:
    conn=pyodbc.connect(connection_string)
    print("SQL server connected successfully :")
    cursor=conn.cursor()
    cursor.execute("SELECT DB_NAME()")
    result=cursor.fetchone()
    print("connected database :", result[0])
    cursor.close()
    conn.close()
except Exception as e:
    print("connection error: ", e)
    

SQL server connected successfully :
connected database : covid_19


In [12]:
# api /url

In [13]:
import requests 

In [14]:
url="https://disease.sh/v3/covid-19/countries"
response=requests.get(url,timeout=30)
print("status code:" , response.status_code)
response.raise_for_status()

status code: 200


In [15]:
import requests
import pandas as pd

url = "https://disease.sh/v3/covid-19/countries"

response = requests.get(url, timeout=30)

response.raise_for_status()

data = response.json()

df = pd.DataFrame(data)

covid_df = df[[
    'country',
    'cases',
    'todayCases',
    'deaths',
    'active',
    'population',
    'updated'
]].copy()

covid_df.columns = [
    'country',
    'cases',
    'TodayCases',
    'deaths',
    'active',
    'population',
    'UpdatedAt'
]

covid_df['UpdatedAt'] = pd.to_datetime(
    covid_df['UpdatedAt'],
    unit='ms',
    utc=True
).dt.tz_localize(None)

display(covid_df.head(10))

print("Total records:", len(covid_df))

,country,cases,TodayCases,deaths,active,population,UpdatedAt
0,Afghanistan,234174,0,7996,15098,40754388,2026-11-19 16:39:07.206
1,Albania,334863,0,3605,1025,2866374,2026-11-19 16:39:07.199
2,Algeria,272010,0,6881,82068,45350148,2026-11-19 16:39:07.202
3,Andorra,48015,0,165,47850,77463,2026-11-19 16:39:07.258
4,Angola,107327,0,1937,1971,35027343,2026-11-19 16:39:07.232
5,Anguilla,3904,0,12,3892,15230,2026-11-19 16:39:07.321
6,Antigua and Barbuda,9106,0,146,6,99509,2026-11-19 16:39:07.308
7,Argentina,10128845,0,130841,746,46010234,2026-11-19 16:39:07.081
8,Armenia,451831,0,8777,7892,2971966,2026-11-19 16:39:07.186
9,Aruba,44224,0,292,1494,107609,2026-11-19 16:39:07.261


Total records: 231


In [16]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=covid_19;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = conn.cursor()

# Insert records
query = """
INSERT INTO covid
(country, cases, TodayCases, deaths,
 active, population, UpdatedAt)
VALUES (?, ?, ?, ?, ?, ?, ?)
"""

values = [
    (
        row.country,
        int(row.cases),
        int(row.TodayCases),
        int(row.deaths),
        int(row.active),
        int(row.population),
        row.UpdatedAt.to_pydatetime()
    )
    for row in covid_df.itertuples(index=False)
]

cursor.executemany(query, values)

conn.commit()

print("COVID data inserted successfully!")
print("Records inserted:", len(values))

cursor.close()
conn.close()

COVID data inserted successfully!
Records inserted: 231
